In [30]:
import os
import bs4
from langchain_classic import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [31]:
# Indexing 

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

In [32]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splits = text_splitter.split_documents(docs)

In [33]:
vectorstore = Chroma.from_documents(documents= splits, embedding = OpenAIEmbeddings(model = "text-embedding-3-small")) 
retriever = vectorstore.as_retriever()

In [34]:
prompt = hub.pull("rlm/rag-prompt")
llm = ChatOpenAI(model_name = "gpt-4o-mini", temperature = 0)

In [35]:
'''
When the database finds the chunks, they come back as complex python objects. 
This simple function extracts just the raw text (page_content) and joins them together with double line breaks so the LLM can read them cleanly 
'''

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [36]:
#Chain 

rag_chain = (
    {"context" : retriever | format_docs, "question" : RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

# Question 
rag_chain.invoke("What is Task Decomposition?")

'Task decomposition is the process of breaking down a larger task into smaller, manageable subgoals or steps. This can be achieved through various methods, including simple prompting to a language model, task-specific instructions, or human inputs. Additionally, a distinct approach involves using an external classical planner with the Planning Domain Definition Language (PDDL) for long-horizon planning.'